# Baseline Machine Learning for LVC Dataset

This notebook evaluates baseline machine learning models using:

- real data only
- repeated train/test splits with seeds: 41, 42, 46
- stratified 5-fold cross-validation on the training set
- holdout evaluation on the real test set


In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neural_network import MLPClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

import warnings
warnings.filterwarnings("ignore")

In [3]:
DATA_PATH = "../data/raw/leish_dataset.csv"
TARGET = "diagnosis"
SEEDS = [41, 42, 46]
TEST_SIZE = 0.30
N_SPLITS = 5
MIN_BREED_COUNT = 10

In [4]:
df = pd.read_csv(DATA_PATH)

# Group rare breeds into 'other'
breed_counts = df["breed_name"].value_counts(dropna=False)
rare_breeds = breed_counts[breed_counts < MIN_BREED_COUNT].index
df["breed_name"] = df["breed_name"].replace(rare_breeds, "other")

print("Shape:", df.shape)
display(df.head())

Shape: (456, 17)


,diagnosis,general_state,ectoparasites,nutritional_state,coat,nails,mucosa_color,muzzle_ear_lesion,lymph_nodes,blepharitis,conjunctivitis,alopecia,bleeding,skin_lesion,muzzle_lip_depigmentation,animal_sex,breed_name
0,negativo,bom,ausente,adequado,normal,normal,normal,presente,normal,ausente,Ausente,ausente,presente,Ausente,ausente,M,SRD
1,negativo,bom,ausente,adequado,normal,normal,normal,ausente,normal,ausente,Ausente,ausente,ausente,Ausente,ausente,M,SRD
2,negativo,bom,leve,adequado,normal,normal,normal,ausente,normal,ausente,Ausente,ausente,presente,Ausente,ausente,M,Poodle
3,negativo,bom,ausente,adequado,normal,leves_moderadas,normal,ausente,normal,ausente,Ausente,ausente,ausente,Ausente,ausente,F,other
4,negativo,bom,leve,leve_moderado,normal,leves_moderadas,normal,ausente,leves_moderadas,ausente,Ausente,ausente,ausente,Ausente,ausente,F,SRD


In [5]:
X = df.drop(columns=[TARGET]).copy()
y = df[TARGET].copy()

le = LabelEncoder()
y_encoded = le.fit_transform(y)

print("Classes:", list(le.classes_))
print("Encoded labels:", np.unique(y_encoded))
print(pd.Series(y_encoded).value_counts())

Classes: ['negativo', 'positivo']
Encoded labels: [0 1]
0    320
1    136
Name: count, dtype: int64


In [6]:
categorical_features = X.select_dtypes(include=["object", "category"]).columns.tolist()
numerical_features = X.select_dtypes(include=["number", "bool"]).columns.tolist()

print("Categorical features:", categorical_features)
print("Numerical features:", numerical_features)

Categorical features: ['general_state', 'ectoparasites', 'nutritional_state', 'coat', 'nails', 'mucosa_color', 'muzzle_ear_lesion', 'lymph_nodes', 'blepharitis', 'conjunctivitis', 'alopecia', 'bleeding', 'skin_lesion', 'muzzle_lip_depigmentation', 'animal_sex', 'breed_name']
Numerical features: []


In [7]:
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

numerical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numerical_transformer, numerical_features),
        ("cat", categorical_transformer, categorical_features)
    ],
    remainder="drop"
)

In [8]:
models = {
    "LR": LogisticRegression(max_iter=1000, random_state=42),
    "SVM": SVC(probability=True, random_state=42),
    "KNN": KNeighborsClassifier(),
    "RF": RandomForestClassifier(random_state=42),
    "GB": GradientBoostingClassifier(random_state=42),
    "MLP": MLPClassifier(max_iter=1000, random_state=42)
}

models

{'LR': LogisticRegression(max_iter=1000, random_state=42),
 'SVM': SVC(probability=True, random_state=42),
 'KNN': KNeighborsClassifier(),
 'RF': RandomForestClassifier(random_state=42),
 'GB': GradientBoostingClassifier(random_state=42),
 'MLP': MLPClassifier(max_iter=1000, random_state=42)}

In [9]:
scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1"
}

In [10]:
cv_results = []
holdout_results = []

for seed in SEEDS:
    print("=" * 80)
    print(f"SEED {seed}")
    print("=" * 80)

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y_encoded,
        test_size=TEST_SIZE,
        stratify=y_encoded,
        random_state=seed
    )

    cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=seed)

    for model_name, model in models.items():
        pipe = Pipeline(steps=[
            ("preprocessor", preprocessor),
            ("clf", model)
        ])

        scores = cross_validate(
            pipe,
            X_train,
            y_train,
            cv=cv,
            scoring=scoring,
            n_jobs=-1,
            return_train_score=False
        )

        cv_results.append({
            "seed": seed,
            "model": model_name,
            "cv_accuracy_mean": np.mean(scores["test_accuracy"]),
            "cv_precision_mean": np.mean(scores["test_precision"]),
            "cv_recall_mean": np.mean(scores["test_recall"]),
            "cv_f1_mean": np.mean(scores["test_f1"]),
            "cv_accuracy_std": np.std(scores["test_accuracy"]),
            "cv_precision_std": np.std(scores["test_precision"]),
            "cv_recall_std": np.std(scores["test_recall"]),
            "cv_f1_std": np.std(scores["test_f1"]),
        })

        pipe.fit(X_train, y_train)
        y_pred = pipe.predict(X_test)

        holdout_results.append({
            "seed": seed,
            "model": model_name,
            "test_accuracy": accuracy_score(y_test, y_pred),
            "test_precision": precision_score(y_test, y_pred),
            "test_recall": recall_score(y_test, y_pred),
            "test_f1": f1_score(y_test, y_pred)
        })

        print(f"{model_name} | CV F1: {np.mean(scores['test_f1']):.4f} | Test F1: {f1_score(y_test, y_pred):.4f}")

SEED 41
LR | CV F1: 0.3213 | Test F1: 0.1724
SVM | CV F1: 0.1471 | Test F1: 0.1250
KNN | CV F1: 0.2243 | Test F1: 0.2000
RF | CV F1: 0.3221 | Test F1: 0.2121
GB | CV F1: 0.3631 | Test F1: 0.2154
MLP | CV F1: 0.3329 | Test F1: 0.2597
SEED 42
LR | CV F1: 0.2035 | Test F1: 0.2800
SVM | CV F1: 0.0000 | Test F1: 0.0000
KNN | CV F1: 0.2179 | Test F1: 0.1724
RF | CV F1: 0.2491 | Test F1: 0.2034
GB | CV F1: 0.2623 | Test F1: 0.3019
MLP | CV F1: 0.3122 | Test F1: 0.2667
SEED 46
LR | CV F1: 0.2336 | Test F1: 0.3448
SVM | CV F1: 0.0174 | Test F1: 0.0000
KNN | CV F1: 0.2184 | Test F1: 0.2500
RF | CV F1: 0.2193 | Test F1: 0.3390
GB | CV F1: 0.2310 | Test F1: 0.4444
MLP | CV F1: 0.2787 | Test F1: 0.4058


In [11]:
cv_results_df = pd.DataFrame(cv_results)
holdout_results_df = pd.DataFrame(holdout_results)

print("CV Results:")
display(cv_results_df)

print("Holdout Results:")
display(holdout_results_df)

CV Results:


,seed,model,cv_accuracy_mean,cv_precision_mean,cv_recall_mean,cv_f1_mean,cv_accuracy_std,cv_precision_std,cv_recall_std,cv_f1_std
0,41,LR,0.692708,0.461172,0.252632,0.321340,0.035782,0.118899,0.102056,0.109693
1,41,SVM,0.692808,0.380000,0.094737,0.147138,0.015485,0.220706,0.069824,0.098634
2,41,KNN,0.676984,0.412165,0.157895,0.224309,0.043671,0.172312,0.057655,0.084699
3,41,RF,0.670734,0.422273,0.263158,0.322083,0.050179,0.140516,0.081536,0.098815
4,41,GB,0.686558,0.455238,0.305263,0.363108,0.046071,0.107859,0.102056,0.106744
5,41,MLP,0.645635,0.390578,0.294737,0.332946,0.055565,0.131247,0.071393,0.086572
6,42,LR,0.652083,0.369242,0.147368,0.203524,0.070603,0.156690,0.061378,0.078438
7,42,SVM,0.686558,0.000000,0.000000,0.000000,0.030833,0.000000,0.000000,0.000000
8,42,KNN,0.636210,0.311810,0.168421,0.217894,0.044878,0.126608,0.051568,0.073688
9,42,RF,0.639683,0.354052,0.200000,0.249111,0.045102,0.086876,0.021053,0.016871


Holdout Results:


,seed,model,test_accuracy,test_precision,test_recall,test_f1
0,41,LR,0.649635,0.294118,0.121951,0.172414
1,41,SVM,0.693431,0.428571,0.073171,0.125000
2,41,KNN,0.649635,0.315789,0.146341,0.200000
3,41,RF,0.620438,0.280000,0.170732,0.212121
4,41,GB,0.627737,0.291667,0.170732,0.215385
5,41,MLP,0.583942,0.277778,0.243902,0.259740
6,42,LR,0.737226,0.777778,0.170732,0.280000
7,42,SVM,0.700730,0.000000,0.000000,0.000000
8,42,KNN,0.649635,0.294118,0.121951,0.172414
9,42,RF,0.656934,0.333333,0.146341,0.203390


In [12]:
cv_summary = (
    cv_results_df.groupby("model")[[
        "cv_accuracy_mean", "cv_precision_mean", "cv_recall_mean", "cv_f1_mean"
    ]]
    .mean()
    .sort_values(by="cv_f1_mean", ascending=False)
)

holdout_summary = (
    holdout_results_df.groupby("model")[[
        "test_accuracy", "test_precision", "test_recall", "test_f1"
    ]]
    .mean()
    .sort_values(by="test_f1", ascending=False)
)

print("CV Summary:")
display(cv_summary)

print("Holdout Summary:")
display(holdout_summary)

CV Summary:


,cv_accuracy_mean,cv_precision_mean,cv_recall_mean,cv_f1_mean
model,,,,
MLP,0.616501,0.345443,0.287719,0.307936
GB,0.649983,0.372170,0.235088,0.285470
RF,0.653125,0.354526,0.214035,0.263491
LR,0.670784,0.397546,0.192982,0.252821
KNN,0.644643,0.336087,0.168421,0.220214
SVM,0.689649,0.143333,0.035088,0.054843


Holdout Summary:


,test_accuracy,test_precision,test_recall,test_f1
model,,,,
GB,0.700730,0.531566,0.235772,0.320572
MLP,0.654501,0.399610,0.260163,0.310735
LR,0.703163,0.553377,0.178862,0.265747
RF,0.664234,0.389630,0.186992,0.251498
KNN,0.649635,0.319244,0.154472,0.207471
SVM,0.695864,0.142857,0.024390,0.041667


In [13]:
best_model_cv = cv_summary.index[0]
best_model_holdout = holdout_summary.index[0]

print("Best model by CV F1:", best_model_cv)
print("Best model by Holdout F1:", best_model_holdout)

Best model by CV F1: MLP
Best model by Holdout F1: GB


In [14]:
cv_results_df.to_csv("../results/tables/baseline_cv_results.csv", index=False)
holdout_results_df.to_csv("../results/tables/baseline_holdout_results.csv", index=False)
cv_summary.to_csv("../results/tables/baseline_cv_summary.csv")
holdout_summary.to_csv("../results/tables/baseline_holdout_summary.csv")

print("Baseline results saved to ../results/tables/")

Baseline results saved to ../results/tables/


## Notes

- This notebook evaluates only real data.
- Preprocessing is included inside the pipeline, preventing data leakage.
- Cross-validation is applied only on the training portion of each seed split.
- Holdout test results are computed on the unseen real test set.
